<a href="https://colab.research.google.com/github/Charlotte11b/Charlotte11b/blob/main/Auto_generate_Calendar_Events.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRACE calendar generator

## Instructions:
1. Click **Run all** at the top of the codes
2. When prompted in the cell below, enter:
   - Participant ID
   - Randomization date in dd/mm/yyyy format
3. The notebook will generate and download the `.ics` file automatically

In [ ]:
participant_id = input("Enter Participant ID: ").strip()
rand_date_str = input("Enter randomization date (dd/mm/yyyy): ").strip()

print("Participant ID entered:", participant_id)
print("Randomization date entered:", rand_date_str)

Enter Participant ID: TESTPT0000
Enter randomization date (dd/mm/yyyy): 15/04/2026
Participant ID entered: TESTPT0000
Randomization date entered: 15/04/2026


# 1. Import relevant libraries and helper functions

In [ ]:
from datetime import datetime, date, timedelta, timezone
from pathlib import Path
import re
import uuid
import shutil

In [ ]:
# Folder where .ics files will be saved
OUTPUT_DIR = Path("/content/trace_ics_output")
OUTPUT_DIR.mkdir(exist_ok=True)

def parse_date(date_str: str) -> date:
    return datetime.strptime(date_str.strip(), "%d/%m/%Y").date()

def sanitize_filename(text: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', "_", text.strip())

def escape_ics_text(text: str) -> str:
    return (
        text.replace("\\", "\\\\")
        .replace(";", r"\;")
        .replace(",", r"\,")
        .replace("\n", r"\n")
    )

def format_date_ics(d: date) -> str:
    return d.strftime("%Y%m%d")

def dtstamp_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print(f"Output folder ready: {OUTPUT_DIR}")

Output folder ready: /content/trace_ics_output


# 2. Event Builder and .ics writer

In [ ]:
def build_all_day_event(summary: str, start_date: date, end_date_exclusive: date, description: str) -> str:
    uid = f"{uuid.uuid4()}@trace-followup"
    return (
        "BEGIN:VEVENT\r\n"
        f"UID:{uid}\r\n"
        f"DTSTAMP:{dtstamp_utc()}\r\n"
        f"SUMMARY:{escape_ics_text(summary)}\r\n"
        f"DTSTART;VALUE=DATE:{format_date_ics(start_date)}\r\n"
        f"DTEND;VALUE=DATE:{format_date_ics(end_date_exclusive)}\r\n"
        f"DESCRIPTION:{escape_ics_text(description)}\r\n"
        "STATUS:CONFIRMED\r\n"
        "END:VEVENT\r\n"
    )

def generate_events(participant_id: str, rand_date: date) -> list[str]:
    pid = participant_id.strip()
    events = []

    # 24–36h follow-up (approximate, since only date is entered)
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - 24-36h follow-up",
            start_date=rand_date + timedelta(days=1),
            end_date_exclusive=rand_date + timedelta(days=2),
            description="Approximate 24-36 hour follow-up window generated from randomization date only.",
        )
    )

    # Day 14–21
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - 1st telephone/email follow-up",
            start_date=rand_date + timedelta(days=14),
            end_date_exclusive=rand_date + timedelta(days=22),
            description="Window: 14-21 days after randomization.",
        )
    )

    # Day 28–35
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - 2nd email follow-up",
            start_date=rand_date + timedelta(days=28),
            end_date_exclusive=rand_date + timedelta(days=36),
            description="Window: 28-35 days after randomization.",
        )
    )

    # Visit 1 window = Day 35–55
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - Visit 1 window",
            start_date=rand_date + timedelta(days=35),
            end_date_exclusive=rand_date + timedelta(days=56),
            description="Visit 1 window: Day 35-55 after randomization.",
        )
    )

    # 2nd telephone follow-up = Day 49–69
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - 2nd telephone follow-up",
            start_date=rand_date + timedelta(days=49),
            end_date_exclusive=rand_date + timedelta(days=70),
            description="Window: Day 49-69 after randomization.",
        )
    )

    # Visit / virtual 2 = Day 60–90
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - Visit/virtual 2 window",
            start_date=rand_date + timedelta(days=60),
            end_date_exclusive=rand_date + timedelta(days=91),
            description="Window: Day 60-90 after randomization.",
        )
    )

    # 3rd telephone/email follow-up = Day 170–190
    events.append(
        build_all_day_event(
            summary=f"TRACE - {pid} - 3rd telephone/email follow-up",
            start_date=rand_date + timedelta(days=170),
            end_date_exclusive=rand_date + timedelta(days=191),
            description="Window: Day 170-190 after randomization.",
        )
    )

    return events

def write_ics_file(participant_id: str, events: list[str]) -> Path:
    safe_id = sanitize_filename(participant_id)
    filepath = OUTPUT_DIR / f"TRACE_{safe_id}_followup.ics"

    calendar_text = (
        "BEGIN:VCALENDAR\r\n"
        "VERSION:2.0\r\n"
        "PRODID:-//TRACE//Follow-up Scheduler//EN\r\n"
        "CALSCALE:GREGORIAN\r\n"
        "METHOD:PUBLISH\r\n"
        + "".join(events) +
        "END:VCALENDAR\r\n"
    )

    with open(filepath, "w", encoding="utf-8", newline="") as f:
        f.write(calendar_text)

    return filepath

print("Functions loaded.")

Functions loaded.


# 3. Enter Participant ID and Randomization Date

In [ ]:
rand_date = parse_date(rand_date_str)
events = generate_events(participant_id, rand_date)
ics_file = write_ics_file(participant_id, events)

print(f"Created: {ics_file}")
print(f"Total events generated: {len(events)}")

Created: /content/trace_ics_output/TRACE_TESTPT0000_followup.ics
Total events generated: 7


# 4. download events and show files

In [ ]:
from google.colab import files

rand_date = parse_date(rand_date_str)
events = generate_events(participant_id, rand_date)
ics_file = write_ics_file(participant_id, events)

print(f"Created file: {ics_file.name}")
print(f"Total events generated: {len(events)}")

files.download(str(ics_file))

Created file: TRACE_TESTPT0000_followup.ics
Total events generated: 7


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>